In [1]:
# ═══════════════════════════════════════════════════════════════════════════════
# IPD-LLM-Agents3 — ML Analysis: Full Pipeline with Hyperparameter Tuning
# File: ml_analysis_updated.py
# Run from: IPD-LLM-Agents3/
# Requirements: scikit-learn, shap, pandas, numpy, matplotlib, seaborn
# ═══════════════════════════════════════════════════════════════════════════════

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid", palette="muted")

from sklearn.ensemble      import (RandomForestRegressor, ExtraTreesRegressor,
                                    GradientBoostingRegressor, AdaBoostRegressor)
from sklearn.linear_model  import Ridge, Lasso, ElasticNet
from sklearn.svm           import SVR
from sklearn.tree          import DecisionTreeRegressor
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose       import ColumnTransformer
from sklearn.pipeline      import Pipeline
from sklearn.model_selection import (KFold, cross_validate, GridSearchCV,
                                      cross_val_score)
from sklearn.metrics       import r2_score, mean_absolute_error, mean_squared_error
import shap

# ─────────────────────────────────────────────────────────────────────────────
# STEP 1: Load and Prepare Data
# ─────────────────────────────────────────────────────────────────────────────

CSV_PATH = "/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/csv_output/enriched_registry.csv"
df = pd.read_csv(CSV_PATH)

print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['mean_group_coop_rate'].describe()}")
print(f"\nModel mix distribution:\n{df['model_mix_label'].value_counts()}")

# Feature definitions
CAT_FEATURES = ["model_mix_label"]           # categorical → One-Hot encoded
NUM_FEATURES = [                              # numeric → StandardScaled
    "temperature",
    "history_window",
    "decision_retries"
]

TARGET = "mean_group_coop_rate"

# Prepare dataframe — cast reset bool to int
df_ml = df[CAT_FEATURES + NUM_FEATURES + [TARGET]].copy()
df_ml["reset_int"] = df["reset_between_episodes"].astype(int)
NUM_FEATURES_EXT = NUM_FEATURES + ["reset_int"]

df_ml = df_ml.dropna()
print(f"\nClean dataset: {df_ml.shape[0]} rows, {len(CAT_FEATURES + NUM_FEATURES_EXT)} features")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 2: Build Preprocessing Pipeline
# ─────────────────────────────────────────────────────────────────────────────

preprocessor = ColumnTransformer(transformers=[
    ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), CAT_FEATURES),
    ("num", StandardScaler(), NUM_FEATURES_EXT),
], remainder="drop")

# Get feature names after one-hot expansion
def get_feature_names(preprocessor, cat_features, num_features):
    ohe = preprocessor.named_transformers_["cat"]
    cat_names = list(ohe.get_feature_names_out(cat_features))
    return cat_names + num_features

# Fit preprocessor and transform
X_raw = df_ml[CAT_FEATURES + NUM_FEATURES_EXT]
y = df_ml[TARGET].values
X = preprocessor.fit_transform(X_raw)
feature_names = get_feature_names(preprocessor, CAT_FEATURES, NUM_FEATURES_EXT)

print(f"\nFeature matrix: {X.shape}")
print(f"Features: {feature_names}")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 3: Define Models WITH Hyperparameter Grids
# ─────────────────────────────────────────────────────────────────────────────

# Cross-validation strategy: 5-fold, shuffled, fixed seed for reproducibility
CV = KFold(n_splits=5, shuffle=True, random_state=42)

MODEL_CONFIGS = {

    # ── Linear / Regularised ──────────────────────────────────────────────────
    "Ridge": {
        "model": Ridge(),
        "grid": {
            "alpha": [0.01, 0.1, 1.0, 10.0, 100.0]
            # alpha: L2 penalty strength. Higher = more regularisation.
            # With 79 rows, modest regularisation (1.0–10.0) usually optimal.
        }
    },
    "Lasso": {
        "model": Lasso(max_iter=10000),
        "grid": {
            "alpha": [0.001, 0.01, 0.05, 0.1, 0.5]
            # Lasso zeros out irrelevant features.
            # Small alpha (0.001–0.01) when most features contribute.
        }
    },
    "ElasticNet": {
        "model": ElasticNet(max_iter=10000),
        "grid": {
            "alpha": [0.001, 0.01, 0.1, 0.5],
            "l1_ratio": [0.2, 0.5, 0.8]
            # l1_ratio=0 → pure Ridge; l1_ratio=1 → pure Lasso
            # l1_ratio=0.5 balances both penalties for correlated one-hot features
        }
    },

    # ── Random Forest ─────────────────────────────────────────────────────────
    "RandomForest": {
        "model": RandomForestRegressor(random_state=42, n_jobs=-1, oob_score=True),
        "grid": {
            "n_estimators": [200, 500],
            # 200 trees usually sufficient; 500 gives marginal improvement
            "max_depth": [None, 8, 12],
            # None=unlimited (overfit risk), 8–12 balances depth vs generalisation
            "min_samples_leaf": [1, 2, 3],
            # Min 2–3 leaves reduces overfit on 79-row dataset
            "max_features": ["sqrt", 0.6]
            # sqrt(9)≈3 features per split: standard RF recommendation
            # 0.6 = 60% of features: slightly more information per tree
        }
    },

    # ── Extra Trees ───────────────────────────────────────────────────────────
    "ExtraTrees": {
        "model": ExtraTreesRegressor(random_state=42, n_jobs=-1),
        "grid": {
            "n_estimators": [200, 500],
            "max_depth": [None, 8, 12],
            "min_samples_leaf": [1, 2, 3],
            "max_features": ["sqrt", 0.6]
        }
    },

    # ── Gradient Boosting ─────────────────────────────────────────────────────
    "GradientBoosting": {
        "model": GradientBoostingRegressor(random_state=42),
        "grid": {
            "n_estimators": [50, 100, 200],
            # Keep low to prevent memorisation on 79 rows
            "learning_rate": [0.05, 0.1, 0.2],
            # Lower LR needs more estimators to converge; trade-off with above
            "max_depth": [2, 3, 4],
            # Shallow trees (2–4) prevent overfit in boosting
            "subsample": [0.7, 0.85, 1.0],
            # Stochastic GBM: use only 70–85% of data per tree
            "min_samples_leaf": [3, 5]
            # Min leaf = 3–5 for 79-row datasets
        }
    },

    # ── AdaBoost ─────────────────────────────────────────────────────────────
    "AdaBoost": {
        "model": AdaBoostRegressor(
            estimator=DecisionTreeRegressor(max_depth=3),
            random_state=42
        ),
        "grid": {
            "n_estimators": [50, 100, 200],
            "learning_rate": [0.5, 1.0, 1.5],
            # AdaBoost LR: higher is fine because base learners are weak (max_depth=3)
            "estimator__max_depth": [2, 3, 4]
            # Depth of the weak learner (decision stump or shallow tree)
        }
    },

    # ── SVR ───────────────────────────────────────────────────────────────────
    "SVR": {
        "model": SVR(),
        "grid": {
            "C": [0.1, 1.0, 10.0, 100.0],
            # C: penalty for errors outside epsilon-tube. Higher C = less tolerance.
            "epsilon": [0.01, 0.05, 0.1, 0.2],
            # Width of no-penalty tube. Larger = smoother, fewer support vectors.
            "kernel": ["rbf", "linear"],
            # rbf: radial basis (non-linear); linear: simple, interpretable
            "gamma": ["scale", "auto"]
            # RBF bandwidth. "scale" = 1/(n_features * X.var()) — usually better.
        }
    },
}

# ─────────────────────────────────────────────────────────────────────────────
# STEP 4: Hyperparameter Tuning + Cross-Validation Leaderboard
# ─────────────────────────────────────────────────────────────────────────────

print("\n" + "="*70)
print("HYPERPARAMETER TUNING + 5-FOLD CV LEADERBOARD")
print("="*70)

results = []
best_estimators = {}

for name, config in MODEL_CONFIGS.items():
    print(f"\nTuning {name}...", flush=True)

    # GridSearchCV: find best hyperparameters via inner CV (5-fold)
    gs = GridSearchCV(
        estimator  = config["model"],
        param_grid = config["grid"],
        cv         = KFold(n_splits=5, shuffle=True, random_state=42),
        scoring    = "r2",
        n_jobs     = -1,
        refit      = True   # refit on full data with best params after search
    )
    gs.fit(X, y)

    best_model = gs.best_estimator_
    best_params = gs.best_params_
    best_cv_r2 = gs.best_score_   # mean CV R² across inner folds

    # Outer CV evaluation with best params (unbiased estimate)
    outer_scores = cross_validate(
        best_model, X, y,
        cv=CV,
        scoring=["r2", "neg_mean_absolute_error", "neg_root_mean_squared_error"],
        return_train_score=True,
        n_jobs=-1
    )

    cv_r2   = outer_scores["test_r2"].mean()
    cv_r2_std = outer_scores["test_r2"].std()
    cv_mae  = -outer_scores["test_neg_mean_absolute_error"].mean()
    cv_rmse = -outer_scores["test_neg_root_mean_squared_error"].mean()
    train_r2 = outer_scores["train_r2"].mean()
    gap      = train_r2 - cv_r2

    # Full-data refit for SHAP (uses the model already refit by GridSearchCV)
    best_model.fit(X, y)
    best_estimators[name] = best_model

    results.append({
        "Model":       name,
        "Best_Params": str(best_params),
        "CV_R2":       round(cv_r2, 4),
        "CV_R2_std":   round(cv_r2_std, 4),
        "CV_MAE":      round(cv_mae, 4),
        "CV_RMSE":     round(cv_rmse, 4),
        "Train_R2":    round(train_r2, 4),
        "Gap":         round(gap, 4),
    })

    print(f"  Best params: {best_params}")
    print(f"  CV R²: {cv_r2:.4f} ± {cv_r2_std:.4f}  |  Train R²: {train_r2:.4f}  |  Gap: {gap:.4f}")
    print(f"  MAE: {cv_mae:.4f}  |  RMSE: {cv_rmse:.4f}")

lb = pd.DataFrame(results).sort_values("CV_R2", ascending=False).reset_index(drop=True)

print("\n" + "="*70)
print("FINAL LEADERBOARD (sorted by CV R²)")
print("="*70)
print(lb[["Model","CV_R2","CV_R2_std","CV_MAE","CV_RMSE","Train_R2","Gap"]].to_string(index=False))

# ─────────────────────────────────────────────────────────────────────────────
# STEP 5: SHAP Analysis on Best Model
# ─────────────────────────────────────────────────────────────────────────────

best_name = lb.iloc[0]["Model"]
best_model = best_estimators[best_name]
print(f"\nBest model: {best_name}")
print(f"Best params: {lb.iloc[0]['Best_Params']}")

# SHAP TreeExplainer for tree-based models; KernelExplainer for others
if best_name in ["RandomForest", "ExtraTrees", "GradientBoosting", "AdaBoost"]:
    explainer = shap.TreeExplainer(best_model)
    shap_values = explainer.shap_values(X)
else:
    explainer = shap.KernelExplainer(best_model.predict, shap.sample(X, 50))
    shap_values = explainer.shap_values(X)

# Mean absolute SHAP per feature
mean_shap = np.abs(shap_values).mean(axis=0)
shap_df = pd.DataFrame({
    "Feature":    feature_names,
    "SHAP_mean":  mean_shap,
    "SHAP_pct":   100 * mean_shap / mean_shap.sum()
}).sort_values("SHAP_mean", ascending=False).reset_index(drop=True)

print("\nSHAP Feature Importance:")
print("="*55)
for _, row in shap_df.iterrows():
    bar = "█" * int(row["SHAP_mean"] / mean_shap.max() * 40)
    print(f"  {row['Feature']:<35} {row['SHAP_mean']:.4f}  {row['SHAP_pct']:5.1f}%  {bar}")

# Group SHAP by category
model_mix_shap = shap_df[shap_df["Feature"].str.startswith("model_mix")]["SHAP_pct"].sum()
config_shap    = shap_df[~shap_df["Feature"].str.startswith("model_mix")]["SHAP_pct"].sum()
print(f"\nModel Identity features: {model_mix_shap:.1f}% of total importance")
print(f"Game Config features:    {config_shap:.1f}% of total importance")

# ─────────────────────────────────────────────────────────────────────────────
# STEP 6: Plots
# ─────────────────────────────────────────────────────────────────────────────

OUTPUT_DIR = "/home/rayudu/My_work1/IPD/work/forge/llm/IPD-LLM-Agents3/csv_output/"

# ── Plot 1: Leaderboard bar chart ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
colors = []
for _, row in lb.iterrows():
    if row["CV_R2"] == lb["CV_R2"].max():
        colors.append("#2ecc71")    # green = winner
    elif row["Gap"] > 0.35:
        colors.append("#e74c3c")    # red = badly overfit
    else:
        colors.append("#3498db")    # blue = normal

bars = ax.barh(lb["Model"][::-1], lb["CV_R2"][::-1], color=colors[::-1],
               edgecolor="white", linewidth=0.8)
ax.axvline(lb["CV_R2"].max(), color="#2ecc71", linestyle="--", alpha=0.6, lw=1.5)
for bar, val in zip(bars, lb["CV_R2"][::-1]):
    ax.text(val + 0.005, bar.get_y() + bar.get_height()/2,
            f"{val:.3f}", va="center", fontsize=9)
ax.set_xlabel("CV R² (5-Fold Cross-Validation)", fontsize=11)
ax.set_title(f"ML Leaderboard — Predicting Group Cooperation Rate\n"
             f"Best: {best_name} (CV R² = {lb['CV_R2'].max():.3f})", fontsize=12)
ax.set_xlim(0, lb["CV_R2"].max() + 0.15)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}ml_leaderboard_tuned_20.png", dpi=150)
plt.close()
print(f"\nSaved: {OUTPUT_DIR}ml_leaderboard_tuned.png")

# ── Plot 2: SHAP bar chart ────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
shap_plot = shap_df.head(10)
bar_colors = ["#e74c3c" if "model_mix" in f else "#3498db"
              for f in shap_plot["Feature"]]
ax.barh(shap_plot["Feature"][::-1], shap_plot["SHAP_mean"][::-1],
        color=bar_colors[::-1], edgecolor="white", linewidth=0.8)
for i, (_, row) in enumerate(shap_plot[::-1].iterrows()):
    ax.text(row["SHAP_mean"] + 0.001,
            i, f"{row['SHAP_pct']:.1f}%", va="center", fontsize=9)
ax.set_xlabel("SHAP Value", fontsize=11)
ax.set_title(f"SHAP Feature Importance — {best_name}\n"
             , fontsize=12)
ax.axvline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}shap_importance_tuned_20.png", dpi=150)
plt.close()
print(f"Saved: {OUTPUT_DIR}shap_importance_tuned.png")

# ── Plot 3: Actual vs Predicted ───────────────────────────────────────────────
y_pred = best_model.predict(X)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y, y_pred, alpha=0.6, edgecolors="white", linewidth=0.5,
           c="#3498db", s=60)
mn, mx = min(y.min(), y_pred.min()), max(y.max(), y_pred.max())
ax.plot([mn, mx], [mn, mx], "r--", lw=1.5, label="Perfect prediction")
ax.set_xlabel("Actual Cooperation Rate", fontsize=11)
ax.set_ylabel("Predicted Cooperation Rate", fontsize=11)
train_r2 = r2_score(y, y_pred)
ax.set_title(f"Actual vs Predicted — {best_name}\n"
             f"Train R²={train_r2:.3f}  |  CV R²={lb['CV_R2'].max():.3f}  |  "
             f"MAE={lb['CV_MAE'].iloc[0]:.3f}", fontsize=11)
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}actual_vs_pred_tuned_20.png", dpi=150)
plt.close()
print(f"Saved: {OUTPUT_DIR}actual_vs_pred_tuned.png")

# ── Plot 4: Overfitting Gap comparison ────────────────────────────────────────
fig, ax = plt.subplots(figsize=(10, 5))
x = np.arange(len(lb))
w = 0.35
b1 = ax.bar(x - w/2, lb["Train_R2"], w, label="Train R²", color="#e74c3c", alpha=0.8)
b2 = ax.bar(x + w/2, lb["CV_R2"],    w, label="CV R²",    color="#2ecc71", alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(lb["Model"], rotation=20, ha="right")
ax.set_ylabel("R²")
ax.set_title("Train R² vs CV R² — Overfitting Gap by Model", fontsize=12)
ax.legend()
ax.axhline(0, color="black", linewidth=0.5)
plt.tight_layout()
plt.savefig(f"{OUTPUT_DIR}overfit_gap_tuned_20.png", dpi=150)
plt.close()
print(f"Saved: {OUTPUT_DIR}overfit_gap_tuned.png")

# ── Plot 5: SHAP beeswarm (if TreeExplainer) ─────────────────────────────────
if best_name in ["RandomForest", "ExtraTrees", "GradientBoosting", "AdaBoost"]:
    fig, ax = plt.subplots(figsize=(10, 6))
    shap.summary_plot(shap_values, X,
                      feature_names=feature_names,
                      show=False, plot_type="dot")
    plt.title(f"SHAP Beeswarm — {best_name}", fontsize=12)
    plt.tight_layout()
    plt.savefig(f"{OUTPUT_DIR}shap_beeswarm_tuned_20.png", dpi=150)
    plt.close()
    print(f"Saved: {OUTPUT_DIR}shap_beeswarm_tuned.png")

print("\nAll plots saved. Analysis complete.")

Dataset shape: (79, 27)
Target stats:
count    79.000000
mean      0.625241
std       0.230722
min       0.151000
25%       0.475000
50%       0.621000
75%       0.768667
max       1.000000
Name: mean_group_coop_rate, dtype: float64

Model mix distribution:
model_mix_label
3L          28
1L+1M+1G    15
2L+1M        9
2L+1G        9
2G+1L        9
3G           9
Name: count, dtype: int64

Clean dataset: 79 rows, 5 features

Feature matrix: (79, 10)
Features: ['model_mix_label_1L+1M+1G', 'model_mix_label_2G+1L', 'model_mix_label_2L+1G', 'model_mix_label_2L+1M', 'model_mix_label_3G', 'model_mix_label_3L', 'temperature', 'history_window', 'decision_retries', 'reset_int']

HYPERPARAMETER TUNING + 5-FOLD CV LEADERBOARD

Tuning Ridge...
  Best params: {'alpha': 0.01}
  CV R²: 0.6443 ± 0.0999  |  Train R²: 0.7683  |  Gap: 0.1240
  MAE: 0.0907  |  RMSE: 0.1277

Tuning Lasso...
  Best params: {'alpha': 0.001}
  CV R²: 0.6457 ± 0.0937  |  Train R²: 0.7672  |  Gap: 0.1215
  MAE: 0.0909  |  RMSE: 0